In [1]:
# Basic libraries of python for numeric and dataframe computations

import numpy as np                              
import pandas as pd
from numpy import asarray
from numpy import *
from sklearn.preprocessing import OrdinalEncoder
from numba import njit

# Basic library for data visualization
import matplotlib.pyplot as plt     

# Slightly advanced library for data visualization            
import seaborn as sns                          

# Featauretools for feature engineering
import featuretools as ft

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

#Importing our ML toolkit
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold, learning_curve

# Importing primitives
from featuretools.primitives import (Minute, Hour, Day, Month,
                                     Weekday, IsWeekend, Count, Sum, Mean, Median, Std, Min, Max)

# Used to ignore the warning given as output of the code
import warnings
warnings.filterwarnings("ignore")

print(ft.__version__)
%load_ext autoreload
%autoreload 2

/opt/conda/lib/python3.7/site-packages/woodwork/__init__.py:23: FutureWarning: Woodwork may not support Python 3.7 in next non-bugfix release.
  "Woodwork may not support Python 3.7 in next non-bugfix release.", FutureWarning
/opt/conda/lib/python3.7/site-packages/featuretools/__init__.py:67: FutureWarning: Featuretools may not support Python 3.7 in next non-bugfix release.
  FutureWarning,


1.11.1


In [2]:
#Creating a function for comprehensive DEA, generic to any dataset
#______________________________________________________________________
#the input contains the dataset in data frame, and an optional column to drop, e.g. the target column
#i.e. the input parameter 'target' is optional

def discover(df, target=''):
    
    #Creating Seris with Feature Types, removing the target feature
    if target != '':
        df1 = df.drop(target, axis = 1, inplace = False)
    else:
        df1 = df

    #listing feature type
    feature_type = []
    for j in range(len(df1.columns)):
        
        if df1[df1.columns[j]].nunique()==2:
            feature_type.append('Binary')
            
            
        elif df1[df1.columns[j]].dtypes != 'O' and df1[df1.columns[j]].nunique()>10:
            feature_type.append('Numerical')
            
        elif df1[df1.columns[j]].dtypes != 'O' and df1[df1.columns[j]].nunique()<10:
            feature_type.append('Ordinal')
                    
        elif df1[df1.columns[j]].nunique()<10:
            feature_type.append('Ordinal')
        
        else:
            feature_type.append('Identifier')
    
    #listing feature content
    features_content = []
    for i in range(len(df1.columns)):
        if df1[df1.columns[i]].nunique()<10: 
            features_content.append(df1[df1.columns[i]].unique())
        else:
            features_content.append('n = ' + str(df1[df1.columns[i]].nunique()))
            
    

            
    #listing empty cells feature type
    num_of_empty = []
    for k in df1.columns:
        num_of_empty.append(df1[k].isnull().sum())

    #listing empty cells ratio feature type
    ratio_of_empty = []
    for k in df1.columns:
        ratio_of_empty.append(round(100*df1[k].isnull().sum()/df1[k].isnull().count(),2))
        
        
    
    
    discovered = pd.DataFrame({'Features': df1.columns, 
                               'Features Content': features_content, 
                               'Feature Type': feature_type, 
                               'Empty Cells': num_of_empty,
                               '% Empty': ratio_of_empty
                             })
    discovered = discovered.sort_values(['Feature Type','Empty Cells'],ascending=False)
    print("Shape of Dataset: ",df.shape)
    return discovered
#_______________________________________________________________________________________________________________
#function to visualize the dataset, works best with classification datasets
#to modify later to take also numerical target data
#the function takes: df: the dataframe containing the dataset, and target: a string containing the column name of the target column (i.e. the one containing the classes)
#_______________________________________________________________________________________________________________

def graph(df, target, spacing = 5):
    
        
    if target != '':
        df1 = df.drop(target, axis = 1, inplace = False)
    else:
        df1 = df

#listing feature types
    feature_type = []
    for j in range(len(df1.columns)):
        
        if df1[df1.columns[j]].nunique()==2:
            feature_type.append('Binary')
            
            
        elif df1[df1.columns[j]].dtypes != 'O' and df1[df1.columns[j]].nunique()>10:
            feature_type.append('Numerical')
            
        elif df1[df1.columns[j]].dtypes != 'O' and df1[df1.columns[j]].nunique()<10:
            feature_type.append('Ordinal')
                    
        elif df1[df1.columns[j]].nunique()<10:
            feature_type.append('Ordinal')
        
        else:
            feature_type.append('Identifier')
            
#listing feature content
    features_content = []
    for i in range(len(df1.columns)):
        if df1[df1.columns[i]].nunique()<10: 
            features_content.append(df1[df1.columns[i]].unique())
        else:
            features_content.append('n = ' + str(df1[df1.columns[i]].nunique()))




#listing empty cells feature type
    num_of_empty = []
    for k in df1.columns:
        num_of_empty.append(df1[k].isnull().sum())




    discovered = pd.DataFrame({'Features': df1.columns, 
                                   'Features Content': features_content, 
                                   'Feature Type': feature_type, 
                                   'Empty Cells': num_of_empty
                                 })

#_________________________________________________________________________________________________    
# Plotting:
#_________________________________________________________________________________________________    

#plotting the target column
    plt.figure(figsize=(10,4))
    plt.subplots_adjust(left=0.1,bottom=0.1, right=0.9,  top=0.9, wspace=0.4, hspace=0.4)
    ax = df[target].value_counts().plot.barh()
    plt.title(f'{target} Count Plot')



#plotting ordinal & binary values:
    
    plt.subplots_adjust(left=0.1,bottom=0.1, right=0.9,  top=0.9, wspace=0.4, hspace=0.4)

    plots = 0
    legends = df[target].unique()
    columns = min(spacing,len(discovered['Features'][discovered['Feature Type'] .isin(['Ordinal', 'Binary'])]))
    rows = int(np.ceil(len(legends)*len(discovered['Features'][discovered['Feature Type'] .isin(['Ordinal', 'Binary'])])/spacing))
    plt.figure(figsize=(20,5*rows))
    
    for i in discovered['Features'][discovered['Feature Type'] .isin(['Ordinal', 'Binary'])]:
        plots = plots + 1
        plt.subplot(rows,columns,plots)
        sns.countplot(x=i, hue=target, data=df, palette="Set1")


#plotting Numerical values:


    plt.subplots_adjust(left=0.1,bottom=0.1, right=0.9,  top=0.9, wspace=0.4, hspace=0.4)

    plots = 0
    columns = min(spacing,len(discovered['Features'][discovered['Feature Type'] =='Numerical']))
    rows = int(np.ceil(len(discovered['Features'][discovered['Feature Type'] =='Numerical'])/spacing))
    plt.figure(figsize=(20,5*rows))
    for i in discovered['Features'][discovered['Feature Type'] =='Numerical']:
        plots = plots + 1
        plt.subplot(rows,columns,plots)
        sns.histplot(data = df, x = df[i], hue = target, hue_order = legends, kde = True, element = 'step')
        #plt.yscale('symlog')
        if df[i].skew(axis = 0, skipna = True)>2:
            plt.xscale('symlog')
            plt.xlim(0,max(df[i]))
            
    plt.show()
    
#Plotting Overall Correlation Heatmap
    
    columns = min(spacing,len(discovered['Features'][discovered['Feature Type'] =='Numerical']))
    rows    = int(np.ceil(len(discovered['Features'][discovered['Feature Type'] =='Numerical'])/spacing))
    plt.figure(figsize=(5*columns,5*rows))
    cmap = sns.diverging_palette(230, 20, as_cmap = True)
    sns.heatmap(df.corr().abs(), annot = True, fmt = '.2f', cmap = cmap )
    plt.show()
    
    


In [4]:
df_train   = pd.read_csv('../input/spaceship-titanic/train.csv',encoding='utf-8')
df_test    = pd.read_csv('../input/spaceship-titanic/test.csv', encoding='utf-8')
submission = pd.read_csv('../input/spaceship-titanic/sample_submission.csv', encoding='utf-8')




In [ ]:
discover(df_train).style.background_gradient(cmap='coolwarm').set_precision(2)

In [ ]:
graph(df_train, 'Transported',5)

In [6]:
#splitting cabin number
combine = [df_train, df_test]

for dataset in combine:
    dataset[['deck', 'num', 'side']] = dataset['Cabin'].str.split("/", expand = True)
    dataset[['Group', 'NumInGroup']] = dataset['PassengerId'].str.split("_", expand = True)
    dataset['NumInGroup']            = dataset['NumInGroup'].astype(int)
    dataset['TotalSpent'] = dataset['ShoppingMall']+dataset['VRDeck']+dataset['FoodCourt']+dataset['Spa']+dataset['RoomService']
    dataset = dataset.drop(columns = ['Cabin','Name', 'num', 'Group', 'PassengerId']   , axis = 1, inplace = True)

In [ ]:
#Imputing Missing Values in Numerical Data

RANDOM_STATE = 12 
FOLDS = 5
STRATEGY = 'median'

imputer_cols = ["Age", "FoodCourt", "ShoppingMall", "Spa", "VRDeck" ,"RoomService"]
imputer = SimpleImputer(strategy=STRATEGY )
imputer.fit(df_train[imputer_cols])
df_train[imputer_cols] = imputer.transform(df_train[imputer_cols])
df_test[imputer_cols] = imputer.transform(df_test[imputer_cols])
df_train["HomePlanet"].fillna('Z', inplace=True)
df_test["HomePlanet"].fillna('Z', inplace=True)



In [ ]:
#Adding feature: total spent

combine = [df_train, df_test]
for dataset in combine:
    dataset['TotalSpent'] = dataset['ShoppingMall']+dataset['VRDeck']+dataset['FoodCourt']+dataset['Spa']+dataset['RoomService']

In [ ]:
#Filling the gaps of non-numerical features, by 'Mode'

combine = [df_train, df_test]
discovered = discover(df_train,'Transported')

for dataset in combine:
    dataset[discovered['Features'][discovered['Feature Type']=='Categorical']] = dataset[discovered['Features'][discovered['Feature Type']=='Categorical']].fillna(dataset.mode().iloc[0])

for dataset in combine:
    dataset[discovered['Features'][discovered['Feature Type']=='Ordinal']] =     dataset[discovered['Features'][discovered['Feature Type']=='Ordinal']]    .fillna(dataset.mode().iloc[0])
    
for dataset in combine:
    dataset[discovered['Features'][discovered['Feature Type']=='Binary']] =     dataset[discovered['Features'][discovered['Feature Type']=='Binary']]    .fillna(dataset.mode().iloc[0])


df_train = combine[0]
df_test = combine[1]

In [ ]:
discover(df_train)

In [ ]:
#Creating Dummy (using drop_first proved slightly faster for the model)
dumFet = [feature for feature in df_test.columns if df_test[feature].nunique()<10]
dumFet.remove('NumInGroup')

dummy_matrix =pd.get_dummies(df_train[dumFet], drop_first=True)
df_train       = df_train      .drop(df_train[dumFet], axis = 1)
df_train       = pd.concat([df_train ,dummy_matrix], axis=1, ignore_index= False)

dummy_matrix =pd.get_dummies(df_test[dumFet], drop_first=True)
df_test       = df_test      .drop(df_test[dumFet], axis = 1)
df_test       = pd.concat([df_test ,dummy_matrix], axis=1, ignore_index= False)






In [ ]:
#mapping the True False, as apparently they do not respond to get_dummies!

combine = [df_train, df_test]
for dataset in combine:
    
    dataset['CryoSleep'] =dataset['CryoSleep'].map({True:1,False:0})
    dataset['VIP'] =dataset['VIP'].map({True:1,False:0})

    

In [ ]:
graph(df_train, 'Transported')

In [ ]:
for dataset in combine:
    dataset = dataset.drop(columns = ['Cabin','Name', 'num', 'Group', 'PassengerId']   , axis = 1, inplace = True)

In [ ]:
discover(df_test)

In [ ]:
df_train.drop(columns = ['PassengerId']   , axis = 1, inplace = True)

In [ ]:
graph(df_train,'Transported')

In [ ]:
Y_train = df_train['Transported']
X_train = df_train.drop(labels = ['Transported'],axis = 1)
Test = df_test.drop(labels = ['PassengerId'],axis = 1)
print(f"X_train shape is = {X_train.shape}" )
print(f"Y_train shape is = {Y_train.shape}" )
print(f"Test shape is = {Test.shape}" )

In [ ]:
from sklearn.preprocessing import StandardScaler
std_scaler = StandardScaler()
X_train = std_scaler.fit_transform(X_train)
Test = std_scaler.transform(Test)
print(f"X_train shape is = {X_train.shape}" )
print(f"Y_train shape is = {Y_train.shape}" )
print(f"Test shape is = {Test.shape}" )

In [ ]:
kfold = StratifiedKFold(n_splits=10)

In [ ]:
# Modeling step Test differents algorithms 
random_state = 2
classifiers = []
classifiers.append(SVC(random_state=random_state))
classifiers.append(DecisionTreeClassifier(random_state=random_state))
classifiers.append(AdaBoostClassifier(DecisionTreeClassifier(random_state=random_state),random_state=random_state,learning_rate=0.1))
classifiers.append(RandomForestClassifier(random_state=random_state))
classifiers.append(ExtraTreesClassifier(random_state=random_state))
classifiers.append(GradientBoostingClassifier(random_state=random_state))
classifiers.append(MLPClassifier(random_state=random_state))
classifiers.append(KNeighborsClassifier())
classifiers.append(LogisticRegression(random_state = random_state))
classifiers.append(LinearDiscriminantAnalysis())

cv_results = []
for classifier in classifiers :
    cv_results.append(cross_val_score(classifier, X_train, y = Y_train, scoring = "accuracy", cv = kfold, n_jobs=4))

cv_means = []
cv_std = []
for cv_result in cv_results:
    cv_means.append(cv_result.mean())
    cv_std.append(cv_result.std())

cv_res = pd.DataFrame({"CrossValMeans":cv_means,"CrossValerrors": cv_std,"Algorithm":["SVC","DecisionTree","AdaBoost",
"RandomForest","ExtraTrees","GradientBoosting","MultipleLayerPerceptron","KNeighboors","LogisticRegression","LinearDiscriminantAnalysis"]})

g = sns.barplot("CrossValMeans","Algorithm",data = cv_res, palette="Set3",orient = "h",**{'xerr':cv_std})
g.set_xlabel("Mean Accuracy")
g = g.set_title("Cross validation scores")

In [ ]:
# Gradient boosting tunning

GBC = GradientBoostingClassifier()
gb_param_grid = {'loss' : ["deviance"],
              'n_estimators' : [100,200,300],
              'learning_rate': [0.1, 0.05, 0.01],
              'max_depth': [4, 8],
              'min_samples_leaf': [100,150],
              'max_features': [0.3, 0.1] 
              }

gsGBC = GridSearchCV(GBC,param_grid = gb_param_grid, cv=kfold, scoring="accuracy", n_jobs= 4, verbose = 1)

gsGBC.fit(X_train,Y_train)

GBC_best = gsGBC.best_estimator_

# Best score
gsGBC.best_score_

In [ ]:
#ExtraTrees 
ExtC = ExtraTreesClassifier()


## Search grid for optimal parameters
ex_param_grid = {"max_depth": [None],
              "max_features": [1, 3, 10],
              "min_samples_split": [2, 3, 10],
              "min_samples_leaf": [1, 3, 10],
              "bootstrap": [False],
              "n_estimators" :[100,300],
              "criterion": ["gini"]}


gsExtC = GridSearchCV(ExtC,param_grid = ex_param_grid, cv=kfold, scoring="accuracy", n_jobs= 4, verbose = 1)

gsExtC.fit(X_train,Y_train)

ExtC_best = gsExtC.best_estimator_

# Best score
gsExtC.best_score_

In [ ]:
# RFC Parameters tunning 
RFC = RandomForestClassifier()


## Search grid for optimal parameters
rf_param_grid = {"max_depth": [None],
              "max_features": [1, 3, 10],
              "min_samples_split": [2, 3, 10],
              "min_samples_leaf": [1, 3, 10],
              "bootstrap": [False],
              "n_estimators" :[100,300],
              "criterion": ["gini"]}


gsRFC = GridSearchCV(RFC,param_grid = rf_param_grid, cv=kfold, scoring="accuracy", n_jobs= 4, verbose = 1)

gsRFC.fit(X_train,Y_train)

RFC_best = gsRFC.best_estimator_

# Best score
gsRFC.best_score_

In [ ]:
### SVC classifier
SVMC = SVC(probability=True)
svc_param_grid = {'kernel': ['rbf'], 
                  'gamma': [ 0.001, 0.01, 0.1, 1],
                  'C': [1, 10, 50, 100,200,300, 1000]}

gsSVMC = GridSearchCV(SVMC,param_grid = svc_param_grid, cv=kfold, scoring="accuracy", n_jobs= 4, verbose = 1)

gsSVMC.fit(X_train,Y_train)

SVMC_best = gsSVMC.best_estimator_

# Best score
gsSVMC.best_score_

In [ ]:
# Adaboost
DTC = DecisionTreeClassifier()

adaDTC = AdaBoostClassifier(DTC, random_state=7)

ada_param_grid = {"base_estimator__criterion" : ["gini", "entropy"],
              "base_estimator__splitter" :   ["best", "random"],
              "algorithm" : ["SAMME","SAMME.R"],
              "n_estimators" :[1,2],
              "learning_rate":  [0.0001, 0.001, 0.01, 0.1, 0.2, 0.3,1.5]}

gsadaDTC = GridSearchCV(adaDTC,param_grid = ada_param_grid, cv=kfold, scoring="accuracy", n_jobs= 4, verbose = 1)

gsadaDTC.fit(X_train,Y_train)

ada_best = gsadaDTC.best_estimator_

gsadaDTC.best_score_


In [ ]:
test_Survived_RFC = pd.Series(gsRFC.best_score_, name="RFC")
test_Survived_ExtC = pd.Series(gsExtC.best_score_, name="ExtC")
test_Survived_SVMC = pd.Series(gsSVMC.best_score_, name="SVC")
test_Survived_AdaC = pd.Series(gsadaDTC.best_score_, name="Ada")
test_Survived_GBC = pd.Series(gsGBC.best_score_, name="GBC")





# Concatenate all classifier results
ensemble_results = pd.concat([test_Survived_RFC,test_Survived_ExtC,test_Survived_AdaC,test_Survived_GBC, test_Survived_SVMC],axis=1)

plt.ylim(ensemble_results.min().mean(), ensemble_results.max().mean())
g= sns.barplot(data = ensemble_results)

for bar in g.patches: 
    g.annotate(format(bar.get_height(), '.4f'),
                   (bar.get_x() + bar.get_width() / 2,
                    bar.get_height()), ha='center', va='center',
                   size=9, xytext=(0, 8),
                   textcoords='offset points')

In [ ]:
#this creates the prediction file with the best estimator, saves it to user's hard drive

votingC = VotingClassifier(estimators=[('rfc', RFC_best), ('extc', ExtC_best),
('svc', SVMC_best), ('adac',ada_best),('gbc',GBC_best)], voting='soft', n_jobs=4)

votingC = votingC.fit(X_train, Y_train)


#making prediction on the test dataset using the tuned model
#drive.mount('/content/gdrive')
#from google.colab import drive

a = np.array(df_test.PassengerId)
b = votingC.predict(Test)
Ibrahim_Seyam_Submission = pd.DataFrame([a,b], index =['PassengerId','Transported']).T
Ibrahim_Seyam_Submission.to_csv("submission.csv",index=False)
